# RMIT Hackathon 2026 - Challenge 1: Multilingual Prompt-Attack Classification
**Model: `FacebookAI/xlm-roberta-base`** (approved list, loaded from the Kaggle Models input)

Plan: fine-tune XLM-R base for 3 classes (BENIGN / JAILBREAK / PROMPT_INJECTION),
class-weighted loss for the imbalanced data, keep the best epoch by validation macro-F1,
tune per-class logit offsets on validation, write `submission.csv`.

In [ ]:
import os, glob, random, time, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# ---- config ---------------------------------------------------------------
MODEL_DIR  = None        # set by hand if auto-discovery below fails, e.g. "/kaggle/input/.../xlm-roberta-base/..."
MAX_LEN    = 256
BATCH      = 16
EPOCHS     = 3
LR         = 2e-5
WEIGHT_POW = 0.5         # class weight = (N / (3 * count)) ** WEIGHT_POW ; 1.0 = full inverse frequency
VAL_FRAC   = 0.10
SEED       = 42
LABELS     = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I        = {l: i for i, l in enumerate(LABELS)}

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "(no GPU - turn on GPU in Settings!)")

def find_file(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input - add the competition data as input"
    return hits[0]

if MODEL_DIR is None:
    cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)]
    cands = [d for d in cands if "xlm-roberta-base" in d.lower().replace("_", "-")]
    assert cands, "xlm-roberta-base not found - Add Input > Models > xlm-roberta-base, or set MODEL_DIR"
    MODEL_DIR = cands[0]
print("model dir:", MODEL_DIR, os.listdir(MODEL_DIR))

In [ ]:
# ---- data -----------------------------------------------------------------
train = pd.read_csv(find_file("train.csv"), encoding="utf-8")
test  = pd.read_csv(find_file("test.csv"),  encoding="utf-8")
sub   = pd.read_csv(find_file("sample_submission.csv"), encoding="utf-8")
print(train.shape, test.shape, sub.shape)
print(train.label.value_counts(normalize=True).round(3).to_dict())

train["y"] = train.label.map(L2I)
strat = train.label + "_" + train.language          # keep label x language mix identical in train/val
tr_df, va_df = train_test_split(train, test_size=VAL_FRAC, random_state=SEED, stratify=strat)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)
print("train/val:", len(tr_df), len(va_df))

counts = tr_df.y.value_counts().sort_index().values
class_w = (len(tr_df) / (3 * counts)) ** WEIGHT_POW
print("class weights:", dict(zip(LABELS, class_w.round(3))))

In [ ]:
# ---- tokenisation (no padding here; each batch is padded to its own max length) ----
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
def encode(texts):
    return tok(list(texts), truncation=True, max_length=MAX_LEN, add_special_tokens=True)["input_ids"]

t0 = time.time()
tr_ids, va_ids, te_ids = encode(tr_df.text), encode(va_df.text), encode(test.text)
print(f"tokenised in {time.time()-t0:.0f}s; truncated share (train): {np.mean([len(x)==MAX_LEN for x in tr_ids]):.1%}")

def make_batch(ids_list, idx, ys=None):
    seqs = [ids_list[i] for i in idx]
    m = max(len(s) for s in seqs)
    inp = torch.full((len(seqs), m), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(seqs), m), dtype=torch.long)
    for r, s in enumerate(seqs):
        inp[r, :len(s)] = torch.tensor(s); att[r, :len(s)] = 1
    out = {"input_ids": inp.to(dev), "attention_mask": att.to(dev)}
    if ys is not None:
        out["labels"] = torch.tensor([ys[i] for i in idx], dtype=torch.long).to(dev)
    return out

@torch.no_grad()
def predict_logits(model, ids_list, bs=64):
    model.eval()
    order = np.argsort([len(x) for x in ids_list])      # sort by length -> much faster inference
    out = np.zeros((len(ids_list), 3), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        b = make_batch(ids_list, idx)
        with torch.cuda.amp.autocast(enabled=dev.type == "cuda"):
            lg = model(**b).logits
        out[idx] = lg.float().cpu().numpy()
    return out

In [ ]:
# ---- model + training -----------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=3).to(dev)
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float).to(dev))
no_decay = ["bias", "LayerNorm.weight"]
params = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
opt = torch.optim.AdamW(params, lr=LR)
steps_per_epoch = (len(tr_ids) + BATCH - 1) // BATCH
sched = get_linear_schedule_with_warmup(opt, int(0.06 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)
scaler = torch.cuda.amp.GradScaler(enabled=dev.type == "cuda")
y_tr, y_va = tr_df.y.tolist(), va_df.y.values

best_f1, best_val_logits, best_test_logits, best_ep = -1, None, None, -1
for ep in range(EPOCHS):
    model.train()
    perm = np.random.permutation(len(tr_ids))
    # bucket loosely by length inside chunks of 50 batches: less padding, still random
    chunks = [perm[i:i + BATCH * 50] for i in range(0, len(perm), BATCH * 50)]
    batches = []
    for c in chunks:
        c = c[np.argsort([len(tr_ids[i]) for i in c])]
        batches += [c[i:i + BATCH] for i in range(0, len(c), BATCH)]
    random.shuffle(batches)
    t0, run = time.time(), 0.0
    for step, idx in enumerate(batches, 1):
        b = make_batch(tr_ids, idx, y_tr)
        labels = b.pop("labels")
        with torch.cuda.amp.autocast(enabled=dev.type == "cuda"):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if step % 200 == 0:
            print(f"ep{ep+1} step {step}/{len(batches)} loss {run/200:.4f} ({time.time()-t0:.0f}s)"); run = 0.0
    vl = predict_logits(model, va_ids)
    f1 = f1_score(y_va, vl.argmax(1), average="macro")
    print(f"== epoch {ep+1}: val macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)")
    if f1 > best_f1:
        best_f1, best_val_logits, best_ep = f1, vl, ep + 1
        best_test_logits = predict_logits(model, te_ids)      # keep test logits of the best epoch (no need to save weights)
print("best epoch:", best_ep, "val macro-F1:", round(best_f1, 4))

In [ ]:
# ---- validation report + per-class logit offsets ----------------------------
print(classification_report(y_va, best_val_logits.argmax(1), target_names=LABELS, digits=4))
for lang in ["pt", "fr", "it", "en"]:
    m = (va_df.language == lang).values
    print(lang, "macro-F1", round(f1_score(y_va[m], best_val_logits[m].argmax(1), average="macro"), 4), "n =", int(m.sum()))

def log_softmax(x):
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))

lv = log_softmax(best_val_logits)
grid = np.arange(-1.5, 1.51, 0.25)
best = (f1_score(y_va, lv.argmax(1), average="macro"), 0.0, 0.0)
for bj in grid:
    for bp in grid:
        f = f1_score(y_va, (lv + np.array([0, bj, bp])).argmax(1), average="macro")
        if f > best[0] + 1e-9: best = (f, bj, bp)
raw_f1 = f1_score(y_va, lv.argmax(1), average="macro")
print(f"raw val F1 {raw_f1:.4f} -> tuned {best[0]:.4f} with offsets JAILBREAK {best[1]}, PROMPT_INJECTION {best[2]}")
# only use the offsets if the gain is not just noise on ~1,400 validation rows
OFFSET = np.array([0, best[1], best[2]]) if best[0] - raw_f1 > 0.003 else np.zeros(3)
print("offsets used:", OFFSET)

In [ ]:
# ---- predict test + write submission ----------------------------------------
pred = (log_softmax(best_test_logits) + OFFSET).argmax(1)
out = pd.DataFrame({"id": test["id"], "label": [LABELS[i] for i in pred]})
out = sub[["id"]].merge(out, on="id", how="left")           # same row order as sample_submission
assert len(out) == 6000 and out.label.isin(LABELS).all() and out.id.is_unique
out.to_csv("/kaggle/working/submission.csv", index=False)
print(out.label.value_counts(normalize=True).round(3).to_dict(), "(train: 62% / 31% / 7%)")
print(out.head())